# Step 3 — The LLM Judge: Vetting the Quiz

Step 2 gave us machine-written multiple-choice questions. But machine-written
means machine-quality: some questions will subtly misread their article. If a
"correct" answer isn't actually supported by the article, Step 4's contestants
get graded against a wrong answer key — and the whole horse race is invalid.

So before any question reaches a contestant, it faces a panel of **three LLM
judges**. Each judge reads the *original article* and the question, and rules
on one thing:

> **faithful** — Is the marked correct option stated or directly supported by
> the article text, with no other option equally defensible?

A question survives only if **at least 2 of the 3 judges** mark it faithful.

This notebook walks the pipeline's three scripts:

| Stage | Script | Output |
|---|---|---|
| Judge (once per model) | `scripts/03-1_generate_judgments.py` | `data/judgments/judgments_<model>.jsonl` |
| Merge to tidy CSV | `scripts/03-2_combine_judgments.py` | `data/judgments/judgments_combined.csv` |
| Seeded selection | `scripts/03-3_select_questions.py` | `data/questions/selected_questions.jsonl` |

We'll make a couple of live judge calls to see the mechanics. The figures
over the scaled-up results live in the companion analysis notebook,
[`03_judgment_analysis.ipynb`](../analysis/03_judgment_analysis.ipynb).

## 1. What the judge sees

The judge gets the article (headline + full text), the question with lettered
options, and which option was marked correct. It does **not** see the
generator's explanation — the question must stand on the article alone.

In [2]:
from toolkit import prompts
from toolkit.utils import load_jsonl

questions = load_jsonl("../../data/questions/questions_gemini-3.1-flash-lite.jsonl")
articles_by_id = {a["id"]: a for a in load_jsonl("../../data/articles/guardian_articles.jsonl")}

question = questions[0]
article = articles_by_id[question["article_id"]]

user_prompt = prompts.build_judge_user_prompt(
    article["headline"],
    article["body_text"],
    question["question"],
    question["options"],
    question["correct_letter"],
)
# The user half: article + question + marked answer (body truncated to peek)
print(user_prompt[:400], "\n   [... full article text ...]\n")
print(user_prompt[user_prompt.find("QUESTION TO JUDGE"):])

ARTICLE HEADLINE: The AI jobs apocalypse probably isn’t coming anytime soon

ARTICLE TEXT:
In March, Anthropic, the cutting-edge artificial intelligence business that gave us the chatbot Claude, published an analysis on the impact of AI on employment, to help us assess the claim that intelligent robots were about to redefine human existence, ending demand for human labor. Last year in May, Anthrop 
   [... full article text ...]

QUESTION TO JUDGE:
According to the analysis published by Anthropic in March, what percentage of tasks in the computer and math category can currently be covered by Claude?

OPTIONS:
A. 10%
B. 33%
C. 50%
D. 100%

MARKED CORRECT ANSWER: B



And the system half is the judge's job description — one binary dimension plus
a rationale:

In [3]:
print(prompts.JUDGE_SYSTEM_PROMPT)

You are an expert auditor of quiz questions. Each question was written from a
specific news article; you will see the article, the question, its options,
and which option was marked correct. Judge the question on one dimension,
answering True or False:

- faithful: The MARKED correct option is stated or directly supported by the
  article text (not hallucinated, not contradicted), and no other option is
  equally defensible given the article.

Also give a 1-2 sentence rationale for your verdict.



## 2. One live judgment

Same structured-output machinery as Step 2, different schema: a single
boolean and a short rationale. The Pydantic class *is* the contract:

```python
class Judgment(BaseModel):
    faithful: bool = Field(description="True if the MARKED correct option is stated or directly supported ...")
    rationale: str = Field(description="1-2 sentences explaining the verdict.")
```

In [4]:
from toolkit.judgments import Judgment
from toolkit.providers import openai_provider

verdict, _raw = openai_provider.run_parsed(
    "gpt-5.4-mini-2026-03-17",       # one of the three judge models
    prompts.JUDGE_SYSTEM_PROMPT,
    user_prompt,
    Judgment,
)

print("Question :", question["question"][:90])
print("Marked   :", question["correct_letter"], "—",
      question["options"]["ABCD".index(question["correct_letter"])][:60])
print()
print("faithful =", verdict.faithful)
print("WHY:", verdict.rationale)

Question : According to the analysis published by Anthropic in March, what percentage of tasks in the
Marked   : B — 33%

faithful = True
WHY: The article explicitly states that Claude covers just 33% of all tasks in the computer and math category. No other option matches the quoted figure, so the marked answer is directly supported.


## 3. Catching a lie

Does the judge actually read, or does it rubber-stamp? Let's frame an
innocent question: keep everything identical but **mark a wrong option as
correct** — exactly the failure mode (a hallucinated answer key) this step
exists to catch:

In [ ]:
wrong_letter = next(l for l in "ABCD" if l != question["correct_letter"])

rigged_prompt = prompts.build_judge_user_prompt(
    article["headline"],
    article["body_text"],
    question["question"],
    question["options"],
    wrong_letter,                     # <- the lie
)
verdict, _raw = openai_provider.run_parsed(
    "gpt-5.4-mini-2026-03-17",
    prompts.JUDGE_SYSTEM_PROMPT,
    rigged_prompt,
    Judgment,
)
print(
    f"Marked {wrong_letter!r} instead of {question['correct_letter']!r} ->",
    "faithful =", verdict.faithful
)

print("WHY:", verdict.rationale)

Marked 'A' instead of 'B' -> faithful = False
WHY: The article explicitly says, “Claude covers just 33% of all tasks in the computer and math category,” so the marked answer A (10%) is contradicted by the text. Option B is directly supported and is the only defensible choice.


## 4. A panel of three judges, at scale

One judge can be wrong — models have blind spots, and a single verdict is a
coin with unknown bias. So we ask **three different models** and take a vote.
Majority voting turns three noisy judges into one robust filter, and
disagreements (as we'll see below) are often the most interesting questions.

Judging hundreds of questions is command-line work, not notebook work — the
same script pattern as Steps 1–2 (threadpool via `--parallel`, crash-safe
appends, resume on question id):

```bash
# 03-1: one run per judge model
for M in gpt-5.6-luna gpt-5.5-2026-04-23 gpt-5.4-mini-2026-03-17; do
  uv run python scripts/03-1_generate_judgments.py \
      --questions data/questions/questions_gemini-3.1-flash-lite.jsonl \
      --articles data/articles/guardian_articles.jsonl \
      --model $M --parallel
done

# 03-2: merge the per-model files into one tidy CSV
uv run python scripts/03-2_combine_judgments.py \
    --input-dir data/judgments --glob 'judgments_*.jsonl'
```

Each judgment row records **both sides of the evaluation** — who wrote the
question (`generator_provider`, `generator_model`) and who judged it
(`judge_model`) — so any later analysis can slice by either.


## 5. The map

| This notebook | Where it lives |
|---|---|
| §1 judge prompts | `toolkit.prompts.JUDGE_SYSTEM_PROMPT`, `build_judge_user_prompt()` |
| §2–3 one judgment | `toolkit.judgments.Judgment`, `judge_question()` |
| §4 panel at scale | `scripts/03-1_generate_judgments.py` (per model) + `toolkit.judgments.judge_questions()`; merged by `scripts/03-2_combine_judgments.py` |

---

### Next up 📊 → 🏁

Run the panel commands above, then open
[`03_judgment_analysis.ipynb`](../analysis/03_judgment_analysis.ipynb) to read
the results — judge strictness, agreement, and the seeded draw that produces
`selected_questions.jsonl`. The demo chain continues with Step 4 in
[`04_answering_methods.ipynb`](04_answering_methods.ipynb).
